In [ ]:
!pip install -q transformers==4.46.0 datasets evaluate seqeval accelerate kagglehub

In [ ]:
import kagglehub
import os
from datasets import Dataset, DatasetDict, Features, Sequence, Value, ClassLabel

# 1. Download
path = kagglehub.dataset_download("alaakhaled/conll003-englishversion")

# 2. Define Helper to read the .txt files
def read_conll_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    data = {"tokens": [], "pos_tags": []}
    sentence_tokens, sentence_pos = [], []
    for line in lines:
        if line.startswith("-DOCSTART-") or line.strip() == "":
            if sentence_tokens:
                data["tokens"].append(sentence_tokens)
                data["pos_tags"].append(sentence_pos)
                sentence_tokens, sentence_pos = [], []
            continue
        splits = line.split()
        if len(splits) >= 2:
            sentence_tokens.append(splits[0])
            sentence_pos.append(splits[1])
    return data

# 3. Process & Create Dataset
pos_list = ['"', "''", '#', '$', '(', ')', ',', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']
train_data = read_conll_file(os.path.join(path, "train.txt"))
val_data = read_conll_file(os.path.join(path, "valid.txt"))

features = Features({"tokens": Sequence(Value("string")), "pos_tags": Sequence(ClassLabel(names=pos_list))})
raw_datasets = DatasetDict({
    "train": Dataset.from_dict(train_data, features=features),
    "validation": Dataset.from_dict(val_data, features=features)
})
label_list = raw_datasets["train"].features["pos_tags"].feature.names
print("Success: Dataset Loaded!")

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["pos_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = raw_datasets.map(tokenize_and_align_labels, batched=True)

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label_list))
data_collator = DataCollatorForTokenClassification(tokenizer)

args = TrainingArguments(
    "distilbert-finetuned-pos",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()

In [ ]:
import evaluate
import numpy as np

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Filtering out -100 (ignored tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Update the trainer with the metrics function and run evaluation
trainer.compute_metrics = compute_metrics
eval_results = trainer.evaluate()

print("\n--- Evaluation Results ---")
for key, value in eval_results.items():
    print(f"{key}: {value}")